# Si Chang shoreline pipeline — Google Colab

รันทีละช่องจากบนลงล่าง (กด ▶ ซ้ายของแต่ละช่อง หรือ **Shift+Enter**)

| ช่อง | ทำอะไร | ต้องรันเมื่อไร |
|---|---|---|
| 1–3 | เชื่อม Drive, ติดตั้งไลบรารี, ล็อกอิน Earth Engine | ทุกครั้งที่เปิด notebook ใหม่ |
| 4 | ตรวจ `sites.json` (งานที่ 1) | ครั้งแรก |
| 5 | ทดสอบด้วยข้อมูลจำลอง | ไม่บังคับ |
| 6–9 | ตารางน้ำ → สำรวจข้อมูล → สั่ง export ไป My Drive/GEE_Sichang | ครั้งเดียว |
| 10 | ดูสถานะ export | รันซ้ำจนครบทุก task |
| 11–12 | วิเคราะห์ทั้งหมด + ดูผล | เมื่อ export เสร็จ / หลังแก้ค่า |
| 13 | แก้ `start_year` / `break_year` แล้วกลับไปรัน 11 | หลังรอบแรก |

**ก่อนเริ่ม:** โฟลเดอร์ `sichang_pipeline` ต้องอยู่ใน **My Drive** (ในเครื่องคือ `G:\My Drive\sichang_pipeline`)

## 1) เชื่อม Google Drive
จะมีหน้าต่างขออนุญาต ให้เลือกบัญชี **kmitl.ac.th** ที่เป็นเจ้าของโฟลเดอร์ แล้วกด Allow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PIPE = '/content/drive/MyDrive/sichang_pipeline'      # <- แก้ถ้าวางโฟลเดอร์ไว้ที่อื่นใน Drive
EXPORTS = '/content/drive/MyDrive/GEE_Sichang'        # ที่ GEE ส่งไฟล์มา
assert os.path.exists(f'{PIPE}/sites.json'), f'ไม่พบ {PIPE}/sites.json — ตรวจว่าโฟลเดอร์อยู่ใน My Drive'
os.chdir(PIPE)
print('OK, working in', os.getcwd())
print(sorted(f for f in os.listdir('.') if f.endswith(('.py', '.json'))))

## 2) ติดตั้งไลบรารี (ประมาณ 1 นาที)

In [ ]:
!pip install -q utide pdfplumber rasterio geopandas earthengine-api
import utide, rasterio, geopandas, ee
print('libraries OK | geopandas', geopandas.__version__, '| earthengine-api', ee.__version__)

## 3) ล็อกอิน Earth Engine
จะมีลิงก์ขออนุญาต ให้เลือกบัญชีเดียวกับที่เปิด project `carbon-vault-513507-e7`

In [ ]:
import json, ee
PROJECT = json.load(open('sites.json', encoding='utf-8'))['ee_project']
ee.Authenticate()
ee.Initialize(project=PROJECT)
print('EE OK:', ee.Number(1).add(1).getInfo(), '| project =', PROJECT)

## 4) ตรวจ `sites.json` (งานที่ 1)

In [ ]:
cfg = json.load(open('sites.json', encoding='utf-8'))
print('ee_project  :', cfg['ee_project'])
print('drive folder:', 'My Drive/' + cfg['drive_folder'])
print('S1          :', cfg['s1'])
print('start_year  :', cfg['start_year'])
import pandas as pd
display(pd.DataFrame(cfg['sites']).T[['type', 'role', 'slope', 'break_year', 'note']])
assert cfg['ee_project'] != 'your-ee-project', 'ยังไม่ได้ใส่ ee_project'
print('\nงานที่ 1 เสร็จ ✅')

## 5) (ไม่บังคับ) ทดสอบด้วยข้อมูลจำลอง
รันในพื้นที่ชั่วคราวของ Colab ไม่แตะไฟล์ใน Drive ผลที่ควรได้: หาด ≈ −0.5 m/yr, rock_control **PASS**, ไซต์ท่าเรือ `coincident = True`

In [ ]:
import shutil, subprocess
T = '/content/selftest'
shutil.rmtree(T, ignore_errors=True)
shutil.copytree(PIPE, T, ignore=shutil.ignore_patterns('results', 'downloads', '*.ipynb'))
!cd {T} && python tests/make_fake_downloads.py . > /dev/null && python run_pipeline.py > run.log 2>&1; tail -n 30 {T}/run.log

## 6) ตารางน้ำเกาะสีชัง
fit ค่าคงที่ฮาร์โมนิกใหม่ด้วย utide เวอร์ชันของ Colab (ไม่กี่วินาที)

In [ ]:
!python 02_fit_tides.py ko_sichang_2026_hourly.csv sichang_tide_coef.pkl

## 7) สำรวจข้อมูล: วงโคจร Sentinel-1 + จำนวนภาพรายปี
บันทึกวงโคจรที่มีภาพมากที่สุดลง `sites.json` ให้อัตโนมัติ

In [ ]:
%run 00_data_inventory.py

## 8) สั่ง export เส้นชายฝั่ง → My Drive/GEE_Sichang
สั่งหลาย task (8 ไซต์ × ช่วงละ 6 ปี × ดาวเทียม) **รันครั้งเดียวพอ** ถ้ารันซ้ำจะได้ไฟล์ซ้ำ (pipeline รวมให้ได้ แต่เปลือง quota)

In [ ]:
%run 04_waterline_gee.py

## 9) สั่ง export การพัฒนาหลังหาดรายปี → My Drive/GEE_Sichang/backshore_annual.csv

In [ ]:
%run 08_backshore_change_gee.py

## 10) ดูสถานะ export
รันซ้ำได้เรื่อยๆ (อาจใช้เวลาหลายสิบนาทีถึงหลายชั่วโมง) ระหว่างนี้ปิด notebook ได้ task ทำงานต่อบนเซิร์ฟเวอร์ของ Google
เมื่อ `RUNNING` และ `READY` เหลือ 0 ให้ไปช่อง 11

In [ ]:
import collections
tasks = [t for t in ee.batch.Task.list() if t.config.get('description', '').startswith(('shoreline_', 'backshore_'))]
st = collections.Counter(t.state for t in tasks)
print('tasks:', dict(st))
for t in tasks:
    if t.state == 'FAILED':
        print('FAILED:', t.config.get('description'), '|', t.status().get('error_message'))
if tasks and st.get('RUNNING', 0) + st.get('READY', 0) == 0:
    print('\nexport เสร็จหมดแล้ว ✅ ไปช่อง 11 ได้')

## 11) วิเคราะห์ทั้งหมด
อ่านไฟล์จาก My Drive/GEE_Sichang โดยตรง แล้วเขียนผลลงโฟลเดอร์ `results/` ใน Drive

In [ ]:
drive.mount('/content/drive', force_remount=True)   # ให้เห็นไฟล์ที่ GEE เพิ่งส่งมา
os.chdir(PIPE)
n = len([f for f in os.listdir(EXPORTS) if f.startswith('shoreline_')]) if os.path.exists(EXPORTS) else 0
print('shoreline files in GEE_Sichang:', n)
!python run_pipeline.py --downloads "{EXPORTS}" 2>&1 | tail -n 60

## 12) ดูผล

In [ ]:
import pandas as pd
s = pd.read_csv('results/site_summary.csv'); display(s)
if os.path.exists('results/driver_timing.csv'):
    display(pd.read_csv('results/driver_timing.csv'))
for site in s.site:
    p = f'results/{site}/scenes_per_year.csv'
    if os.path.exists(p):
        y = pd.read_csv(p, index_col=0)
        thin = y.index[y.total < 2].tolist()
        print(f'{site:16s} scenes/yr after QC: first {y.index.min()}, years with <2 scenes: {thin}')

## 13) ตั้ง `start_year` และ `break_year` แล้วกลับไปรันช่อง 11
- **start_year:** ดูช่อง 12 ถ้าปีต้นๆ มีภาพน้อยกว่า 2 ภาพติดกันหลายปี ให้ตั้งเป็นปีแรกที่ข้อมูลต่อเนื่อง
- **break_year (ไซต์ modified):** ยืนยันจากช่วง 95% (`break_95range`), `dev_pulse_year` และภาพย้อนหลังใน Google Earth แล้วใส่เป็นปีตายตัว
ปล่อย `None` ไว้ถ้ายังไม่ตัดสินใจ

In [ ]:
START_YEAR = None                      # เช่น 1990
BREAK_YEARS = {                        # เช่น 'tha_wang': 2008 ; 'auto' = ให้ระบบหาเอง
    'tha_wang': 'auto',
    'laem_ngu': 'auto',
    'town_harbour': 'auto',
}
cfg = json.load(open('sites.json', encoding='utf-8'))
cfg['start_year'] = START_YEAR
for k, v in BREAK_YEARS.items():
    cfg['sites'][k]['break_year'] = v
json.dump(cfg, open('sites.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
print('saved: start_year =', START_YEAR, '| break_year =', BREAK_YEARS, '-> กลับไปรันช่อง 11')